In [ ]:
!pip install openai

In [1]:
import openai
import json
import time
import hf_token
from openai import OpenAI

In [2]:
client = OpenAI(api_key=hf_token.OPEN_AI_KEY)

In [ ]:
import json
import re

def evaluate_with_chatgpt(question, prediction, references):
    prompt = f"""
You are a strict QA evaluator.

Given:
- Question: {question}
- Reference Answers: {references}
- Model Prediction: {prediction}

Respond ONLY with this exact JSON structure:
{{
  "correct": true or false,
  "rationale": "brief explanation"
}}
"""

    response = client.chat.completions.create(
        model="gpt-4.1-nano",  
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0
    )

    msg = response.choices[0].message.content.strip()
    # print("Raw Response", msg)  

    # To extract the JSON block (more robust than find + slice)
    match = re.search(r"\{\s*\"correct\".*?\}", msg, re.DOTALL)
    if match:
        json_text = match.group(0)
        return json.loads(json_text)

    #if nothing matches
    return {
        "correct": False,
        "rationale": "Model did not return valid JSON."
    }


In [17]:
prediction_files = [
    "finetuned_model_test_results.json",
    "knowledge_graph_model_test_results.json",
    "finetuned_model_train_results.json",
    "knowledge_graph_model_train_results.json",
    "base_model_train_results.json"
]


In [18]:
from tqdm import tqdm

for fname in prediction_files:
    with open(fname, encoding="utf-8") as f:
        records = json.load(f)
    
    #c = 0

    evaluated = []
    print(f"\nEvaluating file: {fname} ({len(records)} records)")

    for rec in tqdm(records, desc=f"Evaluating {fname}"):
        #c += 1

        # if c == 10:
        #     out_name = f"local_evaluated_{fname}"
        #     with open(out_name, "w", encoding="utf-8") as f:
        #         json.dump(evaluated, f, indent=2, ensure_ascii=False)
            
        #     print(f"Wrote {out_name} ({len(evaluated)} entries)")

        #     break
            

        pred = rec.get("predicted_answer")
        verdict = evaluate_with_chatgpt(
            rec["question"],
            pred,
            rec["answers"]
        )
        rec.update(verdict)
        evaluated.append(rec)

    out_name = f"local_evaluated_{fname}"
    with open(out_name, "w", encoding="utf-8") as f:
        json.dump(evaluated, f, indent=2, ensure_ascii=False)

    print(f"Wrote {out_name} ({len(evaluated)} entries)")





Evaluating file: finetuned_model_test_results.json (756 records)


Evaluating finetuned_model_test_results.json: 100%|██████████| 756/756 [10:42<00:00,  1.18it/s]


Wrote local_evaluated_finetuned_model_test_results.json (756 entries)

Evaluating file: knowledge_graph_model_test_results.json (756 records)


Evaluating knowledge_graph_model_test_results.json: 100%|██████████| 756/756 [10:43<00:00,  1.18it/s]


Wrote local_evaluated_knowledge_graph_model_test_results.json (756 entries)

Evaluating file: finetuned_model_train_results.json (3022 records)


...

Wrote local_evaluated_finetuned_model_train_results.json (3022 entries)

Evaluating file: knowledge_graph_model_train_results.json (3022 records)


Evaluating knowledge_graph_model_train_results.json: 100%|██████████| 3022/3022 [39:57<00:00,  1.26it/s] 


Wrote local_evaluated_knowledge_graph_model_train_results.json (3022 entries)

Evaluating file: base_model_train_results.json (3022 records)


Evaluating base_model_train_results.json: 100%|██████████| 3022/3022 [39:16<00:00,  1.28it/s] 

Wrote local_evaluated_base_model_train_results.json (3022 entries)


In [22]:
import json
import os

# List your evaluated JSON files
evaluated_files = [
    "local_evaluated_base_model_test_results.json",
    "local_evaluated_base_model_train_results.json",
    "local_evaluated_finetuned_model_test_results.json",
    "local_evaluated_finetuned_model_train_results.json",
    "local_evaluated_knowledge_graph_model_test_results.json",
    "local_evaluated_knowledge_graph_model_train_results.json",
]

print("Evaluation Accuracy Summary:\n")

for fname in evaluated_files:
    if not os.path.exists(fname):
        print(f"{fname} not found. Skipping.")
        continue

    with open(fname, encoding="utf-8") as f:
        records = json.load(f)

    # print(records[0]["correct"])  # Debugging line to check keys
    total = len(records)
    correct = sum(rec["correct"] for rec in records)
    accuracy = (correct / total) * 100 if total > 0 else 0.0

    print(f"{fname:<55} : {correct}/{total} correct ({accuracy:.2f}%)")


Evaluation Accuracy Summary:

local_evaluated_base_model_test_results.json            : 196/756 correct (25.93%)
local_evaluated_base_model_train_results.json           : 1589/3022 correct (52.58%)
local_evaluated_finetuned_model_test_results.json       : 196/756 correct (25.93%)
local_evaluated_finetuned_model_train_results.json      : 1592/3022 correct (52.68%)
local_evaluated_knowledge_graph_model_test_results.json : 207/756 correct (27.38%)
local_evaluated_knowledge_graph_model_train_results.json : 1580/3022 correct (52.28%)
